In this notebook, we simulate and analyse thermal unfolding curves of a protein dimer obtained at different protein concentrations.
We assume that the underlying model is a two-state reversible unfolding: N2 <-> 2U, where N2 is the native dimer and U is the unfolded monomer.

In this example, higher protein concentration should result in a higher apparent melting temperature.

In [168]:
from pychemelt.utils.plotting import plot_unfolding
from pychemelt.thermal_oligomer import ThermalOligomer
from pychemelt.utils.math import linear_baseline, exponential_baseline

from pychemelt.utils.signals import (
    map_two_state_model_to_signal_fx
)

import numpy as np
from IPython.display import Markdown, display

from notebooks.scripts import display_figure_static

In [169]:
RNG_SEED = 2
CONCS = np.array([1,2,4,8,16,32,64,128])*1e-6 # From 1 µM to 128 µM

DHm_VAL = 200
Tm_VAL = 70
CP0_VAL = 2

INTERCEPT_N = 24
SLOPE_N = -0.27

INTERCEPT_U = 0
PRE_EXP_U = 80.5
EXPONENT_U = 0.0224

def_params = {
    'dHm': DHm_VAL,
    'Tm': Tm_VAL + 273.15,
    'Cp': CP0_VAL,
    'p1_N': 0,
    'p2_N': INTERCEPT_N,
    'p3_N': SLOPE_N,
    'p4_N': 0,
    'p1_U': 0,
    'p2_U': INTERCEPT_U,
    'p3_U': PRE_EXP_U,
    'p4_U': EXPONENT_U,
    'baseline_N_fx':linear_baseline,
    'baseline_U_fx':exponential_baseline

}


def aux_create_pychem_sim(params,concs, model, normalise=False):

    signal_fx = map_two_state_model_to_signal_fx(model)

    # Calculate signal range for proper y-axis scaling
    temp_range  = np.linspace(25, 90, 90-25)
    temp_range_K = temp_range + 273.15

    signal_list = []
    temp_list   = []

    # Use a seeded Generator for reproducible noise in tests
    rng = np.random.default_rng(2)

    for C in concs:

        y = signal_fx(temp_range_K, C, **params)

        # Add gaussian error to signal
        y += rng.normal(0, 0.003*1e-3, len(y)) # Small error (seeded)

        # Simulate variance across positions - by introducing a scaling factor (one per curve)
        factor = np.random.uniform(0.92, 1.08) # Random factor between 0.92 and 1.08 (seeded)

        y *= factor

        signal_list.append(y)
        temp_list.append(temp_range)

    pychem_sim = ThermalOligomer()

    pychem_sim.set_model(model)

    pychem_sim.signal_dic['Fluo'] = signal_list
    pychem_sim.temp_dic['Fluo']   = [temp_range for _ in range(len(concs))]

    pychem_sim.conditions = concs

    pychem_sim.global_min_temp = np.min(temp_range)
    pychem_sim.global_max_temp = np.max(temp_range)

    pychem_sim.set_concentrations()

    pychem_sim.set_signal(['Fluo'])

    pychem_sim.select_conditions(normalise_to_global_max=normalise)
    pychem_sim.expand_multiple_signal()

    pychem_sim.estimate_baseline_parameters(
        native_baseline_type='linear',
        unfolded_baseline_type='exponential',
        window_range_native=12,
        window_range_unfolded=12
    )

    pychem_sim.n_residues = 160  # For cp initial guess - it is important!
    pychem_sim.guess_Cp()

    return pychem_sim

In [170]:
def _extract_core_estimates(params_df):
    if params_df is None or getattr(params_df, 'empty', True):
        return {'Tm (°C)': np.nan, 'dHm': np.nan, 'Cp': np.nan}

    # As requested: use first column values at rows 0, 1, and 2.
    first_col = params_df.iloc[:, 1]
    tm_val = float(first_col.iloc[0])
    dhm_val = float(first_col.iloc[1])
    cp_val = float(first_col.iloc[2])

    return {
        'Tm (°C)': tm_val,
        'ΔH': dhm_val,
        'Cp': cp_val,
    }


fit_comparison_rows = []

In [ ]:
dimer_sim = aux_create_pychem_sim(def_params, CONCS, "Dimer")
fig = plot_unfolding(dimer_sim)
display_figure_static(fig)

In [ ]:
dimer_sim.estimate_derivative()
fig_derivative = plot_unfolding(dimer_sim, plot_derivative=True)
display_figure_static(fig_derivative)

In [ ]:
# Calculate apparent Tms from the first derivative of the unfolding curves
dimer_sim.guess_Tm()
apparent_tms = dimer_sim.t_melting_init_multiple[0]

# plot Tms as a function of concentration
import matplotlib.pyplot as plt
plt.figure(figsize=(8, 5))
plt.scatter(dimer_sim.conditions*1e6, apparent_tms, marker='o')
plt.xlabel('Protein Concentration (µM)')
plt.ylabel('Apparent Tm (°C)')
plt.show()

Now we proceed to fit the data with local intercepts and local slopes.

In [ ]:
dimer_sim.fit_thermal_unfolding_global()
print("Fitted parameters:")
print(dimer_sim.params_df.head())
fit_comparison_rows.append({
    'Fitting strategy': 'Local intercepts + local slopes',
    **_extract_core_estimates(dimer_sim.params_df),
})
fig = plot_unfolding(dimer_sim)
display_figure_static(fig)

Now we proceed to fit the data with local intercepts and global slopes

In [ ]:
dimer_sim.params_df = None
dimer_sim.fit_thermal_unfolding_global()
dimer_sim.fit_thermal_unfolding_global_global()
print("Fitted parameters:")
print(dimer_sim.params_df.head())
fit_comparison_rows.append({
    'Fitting strategy': 'Local intercepts + global slopes',
    **_extract_core_estimates(dimer_sim.params_df),
})
fig = plot_unfolding(dimer_sim)
display_figure_static(fig)

Now we proceed to fit the data with global intercepts and global slopes

In [ ]:
dimer_sim.params_df = None
dimer_sim.fit_thermal_unfolding_global()
dimer_sim.fit_thermal_unfolding_global_global()
dimer_sim.fit_thermal_unfolding_global_global_global(model_scale_factor=True)
print("Fitted parameters:")
print(dimer_sim.params_df.head())
print(dimer_sim.params_df.tail())
fit_comparison_rows.append({
    'Fitting strategy': 'Global intercepts + global slopes',
    **_extract_core_estimates(dimer_sim.params_df),
})
fig = plot_unfolding(dimer_sim)
display_figure_static(fig)

In [ ]:
header = ['Fitting strategy', 'Tm (°C)', 'ΔH', 'Cp']
separator = ['---', '---:', '---:', '---:']

lines = [
    '| ' + ' | '.join(header) + ' |',
    '| ' + ' | '.join(separator) + ' |',
]

for row in fit_comparison_rows:
    tm = row['Tm (°C)']
    dhm = row['ΔH']
    cp = row['Cp']
    tm_txt = f"{tm:.2f}"
    dhm_txt = f"{dhm:.2f}"
    cp_txt = f"{cp:.3f}"
    lines.append(f"| {row['Fitting strategy']} | {tm_txt} | {dhm_txt} | {cp_txt} |")

fit_comparison_markdown = '\n'.join(lines)
display(Markdown('### Comparison table for fitted parameters\n\n' + fit_comparison_markdown))

### Comparison of the three fitting strategies

The table above compares the three fitting modes using the estimated values of $T_m$, $\Delta H$, and $C_p$.

- **Local intercepts + local slopes** is the most flexible model, but has the largest number of parameters.
- **Local intercepts + global slopes** forces all slope terms to be shared, but the intercepts vary freely.
- **Global intercepts + global slopes** has global thermodynamic and baseline parameters.
